# Sentiment Labeling Pipeline

**Steps:**
1. Load `datasets/final/mda_shared_processed.csv`, explode sentences
2. Stratified sample **800 sentences** (by company × year) → Excel for manual labeling
3. Sample **50k sentences** → LLM few-shot labels (inter-rater check)
4. Merge with your manual labels → Cohen's Kappa
   - κ > 0.7 → proceed; else revise prompt in Section 3
5. Label **entire dataset** with LLM
6. Check class proportions (>80% dominant → prompt-refinement placeholder)
7. Spot-check 100 random sentences → Excel
8. Save final labeled dataset

**Labeling guidelines:**
- Clear growth signal / positive factual outcome → **positive**
- Negative factual outcome → **negative**
- Forward-looking statements → **neutral**
- Backward-looking negative statements → **negative**
- Otherwise → **neutral**

## 0. Imports & Config

In [12]:
!pip install ollama

Defaulting to user installation because normal site-packages is not writeable
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.41.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
Using cached pydantic-2.12.5-py3-none-any.whl (463 kB)
Using cached pydantic_core-2.41.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.1 MB)
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [ollama]2m3/5 [pydantic]


In [8]:
import ast
import random
import time
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score
import ollama

# ── paths ──────────────────────────────────────────────────────────────────
ROOT = Path("../")
DATA_PATH          = ROOT / "datasets/final/mda_shared_processed.csv"
MANUAL_OUT         = ROOT / "sentiment_analysis/manual_labeling_sample.xlsx"
MANUAL_LABELED_PATH= ROOT / "sentiment_analysis/manual_labeling_sample_LABELED.xlsx"
FINAL_OUT          = ROOT / "sentiment_analysis/labeled_dataset.csv"
SPOT_CHECK_OUT     = ROOT / "sentiment_analysis/spot_check_100.xlsx"

# Local Ollama model — run `ollama pull llama3.1:8b` once before using
LLM_MODEL = "llama3.1:8b"

VALID_LABELS = {"positive", "negative", "neutral"}
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ModuleNotFoundError: No module named 'ollama'

## 1. Load & Explode Sentences

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f"Raw rows (filings): {len(df_raw):,}")

def parse_sentences(val):
    if isinstance(val, list):
        return val
    try:
        return ast.literal_eval(val)
    except Exception:
        return []

df_raw["sentences_list"] = df_raw["sentences"].apply(parse_sentences)

df = (
    df_raw[["doc_id", "company_name", "filing_type", "filing_date",
            "year", "quarter", "sentences_list"]]
    .explode("sentences_list")
    .rename(columns={"sentences_list": "sentence"})
    .dropna(subset=["sentence"])
    .reset_index(drop=True)
)
df["sentence"] = df["sentence"].str.strip()
df = df[df["sentence"].str.len() > 20].reset_index(drop=True)

print(f"Total sentences: {len(df):,}")
df.head(3)

## 2. Sample 800 Sentences → Excel for Manual Labeling

In [ ]:
MANUAL_N = 800

# Proportional stratified sample across (company_name, year)
group_sizes = df.groupby(["company_name", "year"]).size().reset_index(name="n")
total = group_sizes["n"].sum()
group_sizes["alloc"] = (group_sizes["n"] / total * MANUAL_N).round().astype(int)

# Fix rounding so total == MANUAL_N
diff = MANUAL_N - group_sizes["alloc"].sum()
if diff != 0:
    group_sizes.loc[group_sizes["alloc"].idxmax(), "alloc"] += diff

frames = []
for _, row in group_sizes.iterrows():
    n = int(row["alloc"])
    if n == 0:
        continue
    sub = df[(df["company_name"] == row["company_name"]) & (df["year"] == row["year"])]
    frames.append(sub.sample(n=min(n, len(sub)), random_state=SEED))

manual_sample = (
    pd.concat(frames, ignore_index=True)
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)
manual_sample["manual_label"] = ""  # to be filled by human

manual_sample.to_excel(MANUAL_OUT, index=False)
print(f"Saved {len(manual_sample):,} sentences → {MANUAL_OUT}")
print("Fill in 'manual_label' column (positive / negative / neutral) and save as:")
print(f"  {MANUAL_LABELED_PATH}")
manual_sample[["sentence", "company_name", "year", "manual_label"]].head(5)

## 3. LLM Few-Shot Labeling — 50k Sentences (Inter-rater Check)

In [ ]:
# ── Prompt (edit here if kappa < 0.7) ────────────────────────────────────
FEW_SHOT_EXAMPLES = """
Examples:
Sentence: "Revenue increased 18% year-over-year driven by strong demand across all segments."
Label: positive

Sentence: "Net income declined 32% compared to the prior year due to higher operating costs."
Label: negative

Sentence: "We expect continued growth in our cloud division over the next fiscal year."
Label: neutral

Sentence: "The company experienced a significant decrease in gross margin during fiscal 2022."
Label: negative

Sentence: "We may face increased competition which could adversely affect our market share."
Label: neutral

Sentence: "Operating cash flow improved by $450 million, reflecting disciplined cost management."
Label: positive

Sentence: "The following section discusses our financial condition and results of operations."
Label: neutral
"""

SYSTEM_PROMPT = """You are a financial sentiment classifier for SEC 10-K/10-Q filings.
Classify each sentence as exactly one of: positive, negative, neutral.

Rules:
- Clear growth signal or positive factual outcome (revenue up, profit grew, margin expanded) → positive
- Negative factual outcome (revenue fell, loss incurred, impairment, charges, decline) → negative
- Forward-looking statements (contain 'expect', 'anticipate', 'may', 'could', 'will', 'plan to') → neutral
- Backward-looking negative statements (past-tense deterioration) → negative
- Descriptive, procedural, or ambiguous sentences → neutral

Respond with ONLY the label word."""


def llm_label_batch(sentences, batch_size=20):
    """Label sentences in batches via local Ollama; returns list of label strings."""
    labels = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]
        numbered = "\n".join(f"{j+1}. {s}" for j, s in enumerate(batch))
        user_msg = (
            f"{FEW_SHOT_EXAMPLES}\n"
            "Classify each sentence below. "
            "Return a numbered list matching input order — one label per line.\n\n"
            f"{numbered}"
        )
        for attempt in range(3):
            try:
                resp = ollama.chat(
                    model=LLM_MODEL,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": user_msg},
                    ],
                    options={"temperature": 0},
                )
                raw = resp["message"]["content"].strip()
                parsed = []
                for line in raw.splitlines():
                    line = re.sub(r"^\d+[.)\s]+", "", line.strip()).strip().lower()
                    if line in VALID_LABELS:
                        parsed.append(line)
                # Pad / truncate to match batch length
                parsed = (parsed + ["neutral"] * len(batch))[: len(batch)]
                labels.extend(parsed)
                break
            except Exception as e:
                if attempt == 2:
                    labels.extend(["neutral"] * len(batch))
                else:
                    time.sleep(2 ** attempt)
        if (i // batch_size) % 20 == 0:
            print(f"  ... labeled {i + len(batch):,} / {len(sentences):,}")
    return labels

In [ ]:
LLMCHECK_N = 50_000

# Include all 800 manual sentences so we can compute kappa
manual_idx = set(manual_sample.index)
rest = df.loc[~df.index.isin(manual_idx)]
extra_n = min(LLMCHECK_N - len(manual_sample), len(rest))

llm_check = pd.concat([
    manual_sample.drop(columns=["manual_label"]),
    rest.sample(n=extra_n, random_state=SEED)
], ignore_index=True)

print(f"LLM inter-rater sample: {len(llm_check):,} sentences")
print("Labeling with LLM...")
llm_check["llm_label"] = llm_label_batch(llm_check["sentence"].tolist(), batch_size=50)
print("Done.\n", llm_check["llm_label"].value_counts().to_string())

## 4. Merge Manual Labels & Compute Cohen's Kappa

> **Action required before running this cell:**  
> Open `sentiment_analysis/manual_labeling_sample.xlsx`, fill the `manual_label` column  
> (values: `positive` / `negative` / `neutral`), then save as:  
> `sentiment_analysis/manual_labeling_sample_LABELED.xlsx`

In [ ]:
if not MANUAL_LABELED_PATH.exists():
    raise FileNotFoundError(
        f"Not found: {MANUAL_LABELED_PATH}\n"
        "Complete manual labeling first (see instructions above)."
    )

manual_labeled = pd.read_excel(MANUAL_LABELED_PATH)
manual_labeled["manual_label"] = manual_labeled["manual_label"].str.strip().str.lower()
manual_labeled = manual_labeled[manual_labeled["manual_label"].isin(VALID_LABELS)].copy()
print(f"Valid manually labeled rows: {len(manual_labeled):,}")

merged = manual_labeled.merge(
    llm_check[["sentence", "llm_label"]], on="sentence", how="inner"
)
print(f"Matched for kappa computation: {len(merged):,} rows")

kappa = cohen_kappa_score(merged["manual_label"], merged["llm_label"])
print(f"\nCohen's Kappa = {kappa:.4f}")

if kappa > 0.7:
    print("✓  Good agreement (κ > 0.7) — proceed to full labeling.")
else:
    print("✗  Poor agreement (κ ≤ 0.7) — revise the few-shot prompt in Section 3 and re-run.")

# Confusion matrix
pd.crosstab(
    merged["manual_label"], merged["llm_label"],
    rownames=["manual"], colnames=["llm"], margins=True
)

## 5. Label Entire Dataset

Runs only if kappa > 0.7.

In [ ]:
assert kappa > 0.7, (
    f"Kappa = {kappa:.4f} ≤ 0.7. Revise prompt before proceeding."
)

# Re-use already-labeled sentences to avoid redundant API calls
cached = llm_check.set_index("sentence")["llm_label"].to_dict()
df_full = df.copy()
df_full["llm_label"] = df_full["sentence"].map(cached)

unlabeled = df_full["llm_label"].isna()
to_label = df_full.loc[unlabeled, "sentence"].tolist()
print(f"Cached: {(~unlabeled).sum():,} | Remaining: {len(to_label):,}")

if to_label:
    print("Labeling remaining sentences (may take a while)...")
    df_full.loc[unlabeled, "llm_label"] = llm_label_batch(to_label, batch_size=50)
    print("Done.")

print(f"\nFull dataset: {len(df_full):,} sentences labeled")
df_full["llm_label"].value_counts()

## 6. Check Class Proportions

In [ ]:
props = df_full["llm_label"].value_counts(normalize=True)
print("Class proportions:")
print(props.map("{:.2%}".format).to_string())

dom_cls = props.idxmax()
dom_pct = props.max()

if dom_pct > 0.80:
    print(f"\n⚠️  '{dom_cls}' dominates at {dom_pct:.1%}.")
    print("""
╔══════════════════════════════════════════════════════════════════════╗
║  PROMPT REFINEMENT PLACEHOLDER                                       ║
║                                                                      ║
║  Suggested actions:                                                  ║
║  1. Add more contrastive few-shot examples for minority classes      ║
║  2. Tighten / loosen the dominant-class definition                  ║
║  3. Add a calibration step (e.g. require >0.7 confidence to label)  ║
║  4. Re-run Section 3 with revised prompt, re-check kappa            ║
╚══════════════════════════════════════════════════════════════════════╝
""")
else:
    print(f"\n✓ Class distribution looks healthy (no class > 80%).")

## 7. Spot-Check 100 Random Sentences

In [ ]:
spot = (
    df_full.sample(n=100, random_state=SEED)
    [["sentence", "company_name", "year", "filing_type", "llm_label"]]
    .reset_index(drop=True)
)
spot["reviewer_label"] = ""  # optional: fill in to verify quality

spot.to_excel(SPOT_CHECK_OUT, index=False)
print(f"Spot-check saved → {SPOT_CHECK_OUT}")
spot.head(10)

## 8. Save Final Labeled Dataset

In [ ]:
df_final = (
    df_full[["doc_id", "company_name", "filing_type", "filing_date",
             "year", "quarter", "sentence", "llm_label"]]
    .rename(columns={"llm_label": "sentiment"})
)
df_final.to_csv(FINAL_OUT, index=False)

print(f"Final dataset saved → {FINAL_OUT}")
print(f"Shape: {df_final.shape}")
print("\nLabel distribution:")
print(df_final["sentiment"].value_counts().to_string())
df_final.head(5)